In [1]:
# ===== Step 5.1: environment + dataset/manifest verification =====
!pip -q install open_clip_torch cleanlab 2>/dev/null
import os, glob, torch, pandas as pd, numpy as np
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# --- locate the two datasets ---
roots = {k: sorted(glob.glob(f"/kaggle/input/**/{k}", recursive=True)) for k in
         ["plantdoc-clean-thesis", "plantwild-clean-thesis"]}
for k, v in roots.items(): print(k, "->", v)
PD_ROOT = roots["plantdoc-clean-thesis"][0]
PW_ROOT = roots["plantwild-clean-thesis"][0]

# --- list the CSVs in each ---
for name, r in [("PD", PD_ROOT), ("PW", PW_ROOT)]:
    csvs = sorted(glob.glob(f"{r}/**/*.csv", recursive=True))
    print(f"\n[{name}] {len(csvs)} csv files:")
    for c in csvs: print("  ", os.path.relpath(c, r))

# --- path remapping: find where 'proc/' actually lives in each dataset ---
def find_proc_dir(root):
    hits = sorted(glob.glob(f"{root}/**/proc", recursive=True))
    return hits[0] if hits else root

PD_PROC, PW_PROC = find_proc_dir(PD_ROOT), find_proc_dir(PW_ROOT)
print("\nPD proc dir:", PD_PROC, "\nPW proc dir:", PW_PROC)

def remap(p, proc_dir):
    p = str(p)
    i = p.find("/proc/")
    return os.path.join(proc_dir, p[i+len("/proc/"):]) if i >= 0 else p

# --- load the main manifests (largest csv with proc_path) and check paths ---
def load_manifest(root, proc_dir, prefer):
    cands = [c for c in glob.glob(f"{root}/**/*.csv", recursive=True) if prefer in os.path.basename(c)]
    cands = [c for c in cands if "proc_path" in pd.read_csv(c, nrows=1).columns]
    path = max(cands, key=os.path.getsize)
    df = pd.read_csv(path)
    df["img"] = df["proc_path"].map(lambda p: remap(p, proc_dir))
    ok = df["img"].map(os.path.exists)
    print(f"\n{os.path.basename(path)}: {len(df)} rows | exists {ok.sum()}/{len(df)}")
    if (~ok).any(): print("  missing examples:", df.loc[~ok, "proc_path"].head(3).tolist())
    print("  columns:", list(df.columns))
    for c in [c for c in df.columns if c.endswith("_role")]:
        print(f"  {c}:", df[c].value_counts(dropna=False).to_dict())
    return df

pd_df = load_manifest(PD_ROOT, PD_PROC, "plantdoc")
pw_df = load_manifest(PW_ROOT, PW_PROC, "plantwild")
print("\nStep 5.1 DONE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
torch 2.10.0+cu128 | GPUs: 2 ['Tesla T4', 'Tesla T4']
plantdoc-clean-thesis -> ['/kaggle/input/datasets/mashfii/plantdoc-clean-thesis']
plantwild-clean-thesis -> ['/kaggle/input/datasets/mashfii/plantwild-clean-thesis']

[PD] 4 csv files:
   plantdoc_dinov2b_224_uids.csv
   plantdoc_dup_candidates.csv
   plantdoc_dup_edges_final.csv
   plantdoc_manifest.csv

[PW] 13 csv files:
   audit_summary_table.csv
   pd_pw_class_mapping.csv
   pd_pw_cross_duplicates.csv
   plantdoc_manifest_xd.csv
   plantdoc_threshold_sensitivity.csv
   plantwild_v1_borderline_orb.csv
   plantwild_v1_dinov2b_224_uids.csv
   plantwild_v1_dup_candidates.csv
   plantwild_v1_dup_edges_final.csv
   plantwild_v1_manifest.csv
   plantwild_v1_split_raw.csv
   plantwild_v1_thresho

In [2]:
# ===== Step 5.2: cache DINOv2-B + CLIP ViT-B/16 features for PD + PW =====
import os, glob, time, torch, numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoModel
import open_clip

PD_ROOT = "/kaggle/input/datasets/mashfii/plantdoc-clean-thesis"
PW_ROOT = "/kaggle/input/datasets/mashfii/plantwild-clean-thesis"
PD_PROC, PW_PROC = f"{PD_ROOT}/proc", f"{PW_ROOT}/proc"
OUT = "/kaggle/working/feats"; os.makedirs(OUT, exist_ok=True)

def remap(p, proc_dir):
    p = str(p); i = p.find("/proc/")
    return os.path.join(proc_dir, p[i+6:]) if i >= 0 else p

pd_df = pd.read_csv(glob.glob(f"{PW_ROOT}/**/plantdoc_manifest_xd.csv", recursive=True)[0])
pw_df = pd.read_csv(glob.glob(f"{PW_ROOT}/**/plantwild_v1_manifest.csv", recursive=True)[0], low_memory=False)
pd_df["img"] = pd_df.proc_path.map(lambda p: remap(p, PD_PROC))
pw_df["img"] = pw_df.proc_path.map(lambda p: remap(p, PW_PROC))
print("PD xd cols:", [c for c in pd_df.columns if c.startswith("xd")],
      "| PD exists:", pd_df.img.map(os.path.exists).sum(), "/", len(pd_df))

# shared geometry: resize short side 224 (bicubic) + center crop -> tensor in [0,1]
geo = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
                 T.CenterCrop(224), T.ToTensor()])
class DS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return geo(Image.open(s.p[i]).convert("RGB"))

dev0, dev1 = "cuda:0", ("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
dino = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(dev0).half()
clip, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="laion2b_s34b_b88k")
clip = clip.eval().to(dev1).half()

def norm(x, m, s, dev):
    return ((x.to(dev) - torch.tensor(m, device=dev).view(1,3,1,1)) /
            torch.tensor(s, device=dev).view(1,3,1,1)).half()
IMNET = ([0.485,0.456,0.406], [0.229,0.224,0.225])
CLIPN = ([0.48145466,0.4578275,0.40821073], [0.26862954,0.26130258,0.27577711])

@torch.no_grad()
def extract(df, tag):
    dl = DataLoader(DS(df.img), batch_size=128, num_workers=4, pin_memory=True)
    cls_, mean_, clp = [], [], []; t = time.time()
    for x in dl:
        h = dino(pixel_values=norm(x, *IMNET, dev0)).last_hidden_state.float()
        cls_.append(h[:, 0].cpu()); mean_.append(h[:, 1:].mean(1).cpu())
        clp.append(clip.encode_image(norm(x, *CLIPN, dev1)).float().cpu())
    F = {"dino_cls": torch.cat(cls_), "dino_mean": torch.cat(mean_), "clip": torch.cat(clp)}
    for k, v in F.items():
        v = v.numpy(); assert np.isfinite(v).all(), f"NaN in {tag}_{k}"
        np.save(f"{OUT}/{tag}_{k}.npy", v.astype(np.float32))
        print(f"  {tag}_{k}: {v.shape}  mean|x|={np.abs(v).mean():.3f}")
    df[["uid"]].to_csv(f"{OUT}/{tag}_uids.csv", index=False)
    print(f"  {tag}: {len(df)} imgs in {time.time()-t:.0f}s")

extract(pd_df, "pd")
extract(pw_df, "pw")
pd_df.drop(columns="img").to_csv(f"{OUT}/pd_manifest.csv", index=False)
pw_df.drop(columns="img").to_csv(f"{OUT}/pw_manifest.csv", index=False)

# sanity: nearest-neighbour label agreement on clean train (DINOv2 CLS, cosine)
for tag, df in [("pd", pd_df), ("pw", pw_df)]:
    X = np.load(f"{OUT}/{tag}_dino_cls.npy"); X /= np.linalg.norm(X, axis=1, keepdims=True)
    m = (df.p2_role == "train").values; Xm, y = torch.tensor(X[m]).cuda(), df.class_id.values[m]
    S = Xm @ Xm.T; S.fill_diagonal_(-1); nn = S.argmax(1).cpu().numpy()
    print(f"{tag} 1-NN label agreement (clean train): {(y[nn] == y).mean():.3f}")
print("Files:", sorted(os.listdir(OUT)))
print("Step 5.2 DONE.")

PD xd cols: ['xd_pd2pw_role', 'xd_pw2pd_test_clean'] | PD exists: 2576 / 2576


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

  pd_dino_cls: (2576, 768)  mean|x|=1.362
  pd_dino_mean: (2576, 768)  mean|x|=0.778
  pd_clip: (2576, 512)  mean|x|=0.389
  pd: 2576 imgs in 17s
  pw_dino_cls: (18542, 768)  mean|x|=1.363
  pw_dino_mean: (18542, 768)  mean|x|=0.793
  pw_clip: (18542, 512)  mean|x|=0.388
  pw: 18542 imgs in 100s
pd 1-NN label agreement (clean train): 0.668
pw 1-NN label agreement (clean train): 0.694
Files: ['pd_clip.npy', 'pd_dino_cls.npy', 'pd_dino_mean.npy', 'pd_manifest.csv', 'pd_uids.csv', 'pw_clip.npy', 'pw_dino_cls.npy', 'pw_dino_mean.npy', 'pw_manifest.csv', 'pw_uids.csv']
Step 5.2 DONE.


In [3]:
# ===== Step 5.3: linear-probe teachers on cached features =====
import numpy as np, pandas as pd, torch, torch.nn.functional as F, os, time
from sklearn.metrics import f1_score, accuracy_score
FEAT = "/kaggle/working/feats"; dev = "cuda:0"
OUTP = f"{FEAT}/teacher_probs"; os.makedirs(OUTP, exist_ok=True)

M = {"pd": pd.read_csv(f"{FEAT}/pd_manifest.csv", low_memory=False),
     "pw": pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False)}
def feats(tag):
    L = lambda k: np.load(f"{FEAT}/{tag}_{k}.npy")
    l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)
    dc, dm, cl = L("dino_cls"), L("dino_mean"), L("clip")
    return {"dino_cls": dc, "dino_cls+mean": np.hstack([dc, dm]), "clip": cl,
            "dino+clip": np.hstack([l2(np.hstack([dc, dm])), l2(cl)])}
FE = {t: feats(t) for t in M}

def fit_probe(X, y, n_cls, wd, iters=300):
    X = torch.tensor(X, device=dev); y = torch.tensor(y, device=dev)
    W = torch.zeros(X.shape[1], n_cls, device=dev, requires_grad=True)
    b = torch.zeros(n_cls, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(X @ W + b, y) + wd * (W ** 2).sum()
        loss.backward(); return loss
    opt.step(closure); return W.detach(), b.detach()

def run(tag, role_col, fname, WDS=(1e-5, 1e-4, 1e-3, 1e-2)):
    df = M[tag]; r = df[role_col].values; y = df.class_id.values; K = int(y.max()) + 1
    X = FE[tag][fname]; tr, va, te = r == "train", r == "val", r == "test"
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6; Xs = ((X - mu) / sd).astype(np.float32)
    best = None
    for wd in WDS:
        W, b = fit_probe(Xs[tr], y[tr], K, wd)
        pv = (torch.tensor(Xs[va], device=dev) @ W + b).argmax(1).cpu().numpy()
        f = f1_score(y[va], pv, average="macro")
        if best is None or f > best[0]: best = (f, wd, W, b)
    f_va, wd, W, b = best
    P = torch.softmax(torch.tensor(Xs, device=dev) @ W + b, 1).cpu().numpy()
    pt = P[te].argmax(1)
    return dict(dataset=tag, protocol=role_col, feat=fname, wd=wd, val_f1=f_va,
                test_acc=accuracy_score(y[te], pt), test_f1=f1_score(y[te], pt, average="macro")), P

jobs = [("pd", "p1_role"), ("pd", "p2_role")] + [("pd", f"p3_f{i}_role") for i in range(5)] + \
       [("pw", "p1_role"), ("pw", "p2_role")]
rows, t0 = [], time.time()
for tag, rc in jobs:
    for fn in FE[tag]:
        res, P = run(tag, rc, fn); rows.append(res)
        np.save(f"{OUTP}/{tag}_{rc}_{fn}.npy", P.astype(np.float16))
    print(f"done {tag} {rc}  ({time.time()-t0:.0f}s)")
R = pd.DataFrame(rows); R.to_csv(f"{FEAT}/teacher_probe_results.csv", index=False)

# --- summary table (P3 = mean ± std over 5 folds) ---
R["proto"] = R.protocol.str.replace(r"p3_f\d_role", "p3_role", regex=True)
S = R.groupby(["dataset", "proto", "feat"]).agg(
    val_f1=("val_f1", "mean"), acc=("test_acc", "mean"),
    f1=("test_f1", "mean"), f1_sd=("test_f1", "std")).round(4)
pd.set_option("display.width", 200); print("\n=== TEACHER LINEAR PROBES ===\n", S)

# --- leakage inflation: P1 teacher on official test, twin vs twin-free ---
print("\n=== LEAKAGE INFLATION (P1 teacher, dino+clip) ===")
for tag in ["pd", "pw"]:
    df = M[tag]; te = (df.p1_role == "test").values
    P = np.load(f"{OUTP}/{tag}_p1_role_dino+clip.npy").astype(np.float32)
    ok = (P.argmax(1) == df.class_id.values)
    print(f"{tag} drop_reason on official test:", df.loc[te, "drop_reason"].value_counts(dropna=False).to_dict())
    for name, m in [("all official test", te),
                    ("clean (twin-free, no conflict)", te & (df.in_clean == True).values),
                    ("dropped (twin/conflict)", te & (df.in_clean != True).values)]:
        print(f"  {name:32s} n={m.sum():5d}  acc={ok[m].mean():.4f}")
print("\nStep 5.3 DONE.")

done pd p1_role  (48s)
done pd p2_role  (92s)
done pd p3_f0_role  (130s)
done pd p3_f1_role  (170s)
done pd p3_f2_role  (208s)
done pd p3_f3_role  (245s)
done pd p3_f4_role  (284s)
done pw p1_role  (339s)
done pw p2_role  (394s)

=== TEACHER LINEAR PROBES ===
                                val_f1     acc      f1   f1_sd
dataset proto   feat                                         
pd      p1_role clip           0.7357  0.6822  0.6644     NaN
                dino+clip      0.7964  0.7585  0.7492     NaN
                dino_cls       0.7875  0.7542  0.7464     NaN
                dino_cls+mean  0.7618  0.7500  0.7448     NaN
        p2_role clip           0.6923  0.7045  0.6912     NaN
                dino+clip      0.7807  0.7864  0.7717     NaN
                dino_cls       0.7742  0.7955  0.7801     NaN
                dino_cls+mean  0.7681  0.8000  0.7877     NaN
        p3_role clip           0.6975  0.7305  0.7100  0.0082
                dino+clip      0.7721  0.7916  0.7754  0.

In [4]:
# ===== Step 5.4: leakage inflation via dup_group twins =====
import numpy as np, pandas as pd, torch
FEAT = "/kaggle/working/feats"; OUTP = f"{FEAT}/teacher_probs"
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)

rows = []
for tag, df in M.items():
    r, y = df.p1_role.values, df.class_id.values
    g = df.dup_group.astype(str).values
    trv_groups = set(g[(r == "train") | (r == "val")])
    te = r == "test"
    twin = te & np.isin(g, list(trv_groups)) & (df.dup_group.notna().values)
    conflict = (df.drop_reason == "label_conflict").values
    free = te & ~twin & ~conflict
    twin_nc = twin & ~conflict
    print(f"\n[{tag}] official test {te.sum()} | twin-in-train {twin.sum()} "
          f"({100*twin.sum()/te.sum():.2f}%) | twin & no-conflict {twin_nc.sum()} | twin-free & no-conflict {free.sum()}")

    # teacher (P1 probe, dino+clip)
    P = np.load(f"{OUTP}/{tag}_p1_role_dino+clip.npy").astype(np.float32)
    ok_t = P.argmax(1) == y
    # 1-NN memoriser on P1 train+val (dino+clip features, cosine)
    X = np.hstack([l2(np.hstack([np.load(f"{FEAT}/{tag}_dino_cls.npy"), np.load(f"{FEAT}/{tag}_dino_mean.npy")])),
                   l2(np.load(f"{FEAT}/{tag}_clip.npy"))]); X = l2(X)
    bank = (r == "train") | (r == "val")
    Xb, yb = torch.tensor(X[bank]).cuda(), y[bank]
    Xq = torch.tensor(X[te]).cuda()
    sim, idx = (Xq @ Xb.T).max(1)
    pred_nn = np.full(len(df), -1); pred_nn[te] = yb[idx.cpu().numpy()]
    nn_sim = np.full(len(df), np.nan); nn_sim[te] = sim.cpu().numpy()
    ok_n = pred_nn == y

    for name, m in [("twin-in-train", twin_nc), ("twin-free", free), ("all official", te)]:
        rows.append(dict(dataset=tag, subset=name, n=int(m.sum()),
                         teacher_acc=ok_t[m].mean(), nn1_acc=ok_n[m].mean(),
                         nn_cos_median=np.nanmedian(nn_sim[m])))
    df["p1_test_twin"] = twin  # save flag for later protocols
    df.to_csv(f"{FEAT}/{tag}_manifest.csv", index=False)

T = pd.DataFrame(rows).round(4); pd.set_option("display.width", 200)
print("\n=== LEAKAGE INFLATION (label-conflict images excluded) ===\n", T.to_string(index=False))
for tag in ["pd", "pw"]:
    a = T[(T.dataset == tag)].set_index("subset")
    print(f"{tag}: teacher gap twin - free = {100*(a.loc['twin-in-train','teacher_acc']-a.loc['twin-free','teacher_acc']):+.1f} pp | "
          f"1-NN gap = {100*(a.loc['twin-in-train','nn1_acc']-a.loc['twin-free','nn1_acc']):+.1f} pp")
T.to_csv(f"{FEAT}/leakage_inflation.csv", index=False)
print("\nStep 5.4 DONE.")


[pd] official test 236 | twin-in-train 27 (11.44%) | twin & no-conflict 11 | twin-free & no-conflict 209

[pw] official test 3677 | twin-in-train 414 (11.26%) | twin & no-conflict 241 | twin-free & no-conflict 3229

=== LEAKAGE INFLATION (label-conflict images excluded) ===
 dataset        subset    n  teacher_acc  nn1_acc  nn_cos_median
     pd twin-in-train   11       1.0000   1.0000         0.9760
     pd     twin-free  209       0.7943   0.6603         0.8336
     pd  all official  236       0.7585   0.6314         0.8435
     pw twin-in-train  241       0.9585   0.9917         0.9527
     pw     twin-free 3229       0.7965   0.7241         0.8333
     pw  all official 3677       0.7813   0.7098         0.8417
pd: teacher gap twin - free = +20.6 pp | 1-NN gap = +34.0 pp
pw: teacher gap twin - free = +16.2 pp | 1-NN gap = +26.8 pp

Step 5.4 DONE.


In [5]:
# ===== Step 5.5: OOF teacher probs + cleanlab noise detection vs expert decisions =====
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from cleanlab.rank import get_label_quality_scores
FEAT = "/kaggle/working/feats"; dev = "cuda:0"
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
R = pd.read_csv(f"{FEAT}/teacher_probe_results.csv")
l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)

def X_of(tag):
    L = lambda k: np.load(f"{FEAT}/{tag}_{k}.npy")
    return np.hstack([l2(np.hstack([L("dino_cls"), L("dino_mean")])), l2(L("clip"))]).astype(np.float32)

def fit_probe(X, y, K, wd):
    X = torch.tensor(X, device=dev); y = torch.tensor(y, device=dev)
    W = torch.zeros(X.shape[1], K, device=dev, requires_grad=True); b = torch.zeros(K, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=300, line_search_fn="strong_wolfe")
    def c():
        opt.zero_grad(); l = F.cross_entropy(X @ W + b, y) + wd * (W**2).sum(); l.backward(); return l
    opt.step(c); return W.detach(), b.detach()

def auc_block(name, y_true, score):
    m = ~pd.isna(y_true); yt = np.asarray(y_true[m], dtype=int); s = -score[m]  # low quality = noisy
    if yt.sum() == 0 or yt.sum() == len(yt): return
    print(f"  {name:45s} n={m.sum():6d} pos={yt.sum():5d}  AUROC={roc_auc_score(yt, s):.3f}  "
          f"AP={average_precision_score(yt, s):.3f} (base {yt.mean():.3f})")

for tag, df in M.items():
    X, y = X_of(tag), df.class_id.values; K = int(y.max()) + 1
    wd = R[(R.dataset == tag) & (R.protocol == "p2_role") & (R.feat == "dino+clip")].wd.iloc[0]
    g = df.dup_group.astype(str).where(df.dup_group.notna(), "solo_" + df.uid.astype(str)).values
    oof = np.zeros((len(df), K), np.float32)
    for k, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, y, g)):
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
        W, b = fit_probe((X[tr] - mu) / sd, y[tr], K, wd)
        oof[te] = torch.softmax(torch.tensor((X[te] - mu) / sd, device=dev) @ W + b, 1).cpu().numpy()
    np.save(f"{FEAT}/teacher_probs/{tag}_oof_dino+clip.npy", oof.astype(np.float16))
    q = get_label_quality_scores(y, oof, method="self_confidence")
    df["oof_pred"], df["oof_conf_given"], df["label_quality"] = oof.argmax(1), oof[np.arange(len(y)), y], q
    print(f"\n[{tag}] wd={wd}  OOF acc={np.mean(oof.argmax(1) == y):.4f}  "
          f"(clean only {np.mean((oof.argmax(1) == y)[df.in_clean == True]):.4f})")

    print(" Noise detection (score = cleanlab self-confidence):")
    auc_block("label_conflict flag (ours)", (df.drop_reason == "label_conflict").astype(float).values, q)
    if tag == "pw":
        print("  expert_status counts:", df.expert_status.value_counts(dropna=False).to_dict())
        es = df.expert_status.astype(str).str.lower()
        has = df.expert_status.notna().values
        rel = np.where(has, es.str.contains("relab").astype(float), np.nan)
        rem = np.where(has, es.str.contains("remov").astype(float), np.nan)
        any_ = np.where(has, ((es.str.contains("relab")) | (es.str.contains("remov"))).astype(float), np.nan)
        auc_block("expert: relabelled vs rest", rel, q)
        auc_block("expert: removed vs rest", rem, q)
        auc_block("expert: relabelled|removed vs kept", any_, q)
        # does the teacher's predicted class match the expert's new label on relabelled imgs?
        if "v2_class" in df:
            rl = (rel == 1)
            print(f"  relabelled imgs: teacher OOF pred == our conflict-partner? see next step; "
                  f"oof_conf_given median relabelled={np.median(df.oof_conf_given[rl]):.3f} "
                  f"vs kept={np.median(df.oof_conf_given[(any_ == 0)]):.3f}")
    df.to_csv(f"{FEAT}/{tag}_manifest.csv", index=False)
print("\nStep 5.5 DONE.")


[pd] wd=0.01  OOF acc=0.7822  (clean only 0.7937)
 Noise detection (score = cleanlab self-confidence):
  label_conflict flag (ours)                    n=  2576 pos=   94  AUROC=0.769  AP=0.110 (base 0.036)

[pw] wd=0.0001  OOF acc=0.7632  (clean only 0.7816)
 Noise detection (score = cleanlab self-confidence):
  label_conflict flag (ours)                    n= 18542 pos= 1130  AUROC=0.757  AP=0.195 (base 0.061)
  expert_status counts: {'not_in_v2_classes': 7957, 'kept': 6298, 'removed': 3938, 'relabelled': 349}
  expert: relabelled vs rest                    n= 18542 pos=  349  AUROC=0.820  AP=0.084 (base 0.019)
  expert: removed vs rest                       n= 18542 pos= 3938  AUROC=0.683  AP=0.355 (base 0.212)
  expert: relabelled|removed vs kept            n= 18542 pos= 4287  AUROC=0.706  AP=0.420 (base 0.231)
  relabelled imgs: teacher OOF pred == our conflict-partner? see next step; oof_conf_given median relabelled=0.048 vs kept=0.987

Step 5.5 DONE.


In [6]:
# ===== Step 5.6: corrected expert AUROC + thesis figures for Step 5 =====
import numpy as np, pandas as pd, os, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
FEAT = "/kaggle/working/feats"; FIG = "/kaggle/working/figures"; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight"})
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
LOG = f"{FIG}/figure_log.csv"
log = pd.read_csv(LOG).to_dict("records") if os.path.exists(LOG) else []
def save(fig, name, caption):
    for ext in ["png", "pdf"]: fig.savefig(f"{FIG}/{name}.{ext}", dpi=300)
    plt.close(fig); log[:] = [r for r in log if r["file"] != name] + [dict(file=name, step="5", caption=caption)]
    print("saved", name)

# ---------- corrected noise-detection AUROC (only images the experts judged) ----------
pw = M["pw"]; q = pw.label_quality.values; es = pw.expert_status.values
judged = np.isin(es, ["kept", "relabelled", "removed"])
rows = []
def auc(name, pos_mask, neg_mask, s=q, ds="pw"):
    m = pos_mask | neg_mask; yt = pos_mask[m].astype(int); sc = -s[m]
    r = dict(dataset=ds, comparison=name, n=int(m.sum()), pos=int(yt.sum()),
             AUROC=roc_auc_score(yt, sc), AP=average_precision_score(yt, sc), base=yt.mean())
    rows.append(r); return yt, sc
curves = {}
curves["Relabelled vs kept"] = auc("relabelled vs kept", es == "relabelled", es == "kept")
curves["Removed vs kept"] = auc("removed vs kept", es == "removed", es == "kept")
curves["Relabelled|removed vs kept"] = auc("relabelled|removed vs kept", np.isin(es, ["relabelled", "removed"]), es == "kept")
conf_pw = (pw.drop_reason == "label_conflict").values
curves["PW label-conflict flag"] = auc("label_conflict vs rest", conf_pw, ~conf_pw)
pdm = M["pd"]; conf_pd = (pdm.drop_reason == "label_conflict").values
curves["PD label-conflict flag"] = auc("label_conflict vs rest", conf_pd, ~conf_pd, s=pdm.label_quality.values, ds="pd")
# does our conflict flag agree with experts? relabel rate among judged
print("Relabel rate | conflict:", (es[judged & conf_pw] == "relabelled").mean().round(3),
      "| non-conflict:", (es[judged & ~conf_pw] == "relabelled").mean().round(3))
N = pd.DataFrame(rows).round(4); N.to_csv(f"{FIG}/table_noise_detection.csv", index=False)
print("\n=== NOISE DETECTION (corrected) ===\n", N.to_string(index=False))

# ---------- Fig 5.1 teacher linear probes ----------
R = pd.read_csv(f"{FEAT}/teacher_probe_results.csv")
R["proto"] = R.protocol.str.replace(r"p3_f\d_role", "p3_role", regex=True)
S = R.groupby(["dataset", "proto", "feat"]).test_f1.agg(["mean", "std"]).reset_index()
S.to_csv(f"{FIG}/table_teacher_probes.csv", index=False)
feats_ = ["clip", "dino_cls", "dino_cls+mean", "dino+clip"]
groups = [("pd", "p1_role", "PlantDoc P1"), ("pd", "p2_role", "PlantDoc P2"), ("pd", "p3_role", "PlantDoc P3 (5-fold)"),
          ("pw", "p1_role", "PlantWild P1"), ("pw", "p2_role", "PlantWild P2")]
fig, ax = plt.subplots(figsize=(9, 3.6)); w = 0.2
for i, f in enumerate(feats_):
    vals = [S[(S.dataset == d) & (S.proto == p) & (S.feat == f)] for d, p, _ in groups]
    ax.bar(np.arange(len(groups)) + (i - 1.5) * w, [v["mean"].iloc[0] for v in vals], w,
           yerr=[np.nan_to_num(v["std"].iloc[0]) for v in vals], capsize=2, label=f)
ax.set_xticks(range(len(groups))); ax.set_xticklabels([g[2] for g in groups]); ax.set_ylim(0.5, 0.85)
ax.set_ylabel("Test macro-F1"); ax.legend(ncol=4, fontsize=8, loc="upper center"); ax.grid(axis="y", alpha=.3)
save(fig, "fig5_1_teacher_probes", "Linear-probe foundation teachers: test macro-F1 by feature set and protocol (P3: mean ± SD over 5 folds).")

# ---------- Fig 5.2 leakage inflation with Wilson CIs ----------
L = pd.read_csv(f"{FEAT}/leakage_inflation.csv"); L = L[L.subset != "all official"]
def wilson(p, n, z=1.96):
    c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n); return c - h, c + h
fig, axs = plt.subplots(1, 2, figsize=(8, 3.3), sharey=True)
for ax, tag, title in zip(axs, ["pd", "pw"], ["PlantDoc", "PlantWild"]):
    d = L[L.dataset == tag].set_index("subset")
    for j, (col, lab) in enumerate([("teacher_acc", "Linear-probe teacher"), ("nn1_acc", "1-NN (memoriser)")]):
        for k, sub in enumerate(["twin-free", "twin-in-train"]):
            p, n = d.loc[sub, col], d.loc[sub, "n"]; lo, hi = wilson(p, n)
            ax.bar(j*3 + k, p, color=["#4C72B0", "#DD8452"][k], yerr=[[p-lo], [hi-p]], capsize=3,
                   label=(f"{sub} test" if j == 0 else None))
            ax.text(j*3 + k, 0.03, f"n={n}", ha="center", fontsize=8, color="white")
    ax.set_xticks([0.5, 3.5]); ax.set_xticklabels(["Teacher", "1-NN"]); ax.set_title(title); ax.set_ylim(0, 1.05)
axs[0].set_ylabel("Accuracy (95% Wilson CI)"); axs[0].legend(fontsize=8, loc="lower right")
save(fig, "fig5_2_leakage_inflation", "Accuracy on official-test images with vs without a near-duplicate twin in train (label-conflict images excluded).")

# ---------- Fig 5.3 ROC curves for noise detection ----------
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for name, (yt, sc) in curves.items():
    fpr, tpr, _ = roc_curve(yt, sc); ax.plot(fpr, tpr, label=f"{name} ({roc_auc_score(yt, sc):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(fontsize=7, loc="lower right"); ax.set_title("Label-noise detection (teacher self-confidence)")
save(fig, "fig5_3_noise_roc", "ROC of OOF foundation-teacher label-quality scores against expert v1→v2 decisions and our conflict flag.")

# ---------- Fig 5.4 confidence in given label by expert decision ----------
fig, ax = plt.subplots(figsize=(5.5, 3.3)); bins = np.linspace(0, 1, 26)
for st, c in [("kept", "#4C72B0"), ("removed", "#8C8C8C"), ("relabelled", "#C44E52")]:
    ax.hist(pw.oof_conf_given[es == st], bins, density=True, histtype="step", lw=1.8, color=c,
            label=f"{st} (n={(es == st).sum()}, median={np.median(pw.oof_conf_given[es == st]):.2f})")
ax.set_xlabel("OOF teacher probability of the given label"); ax.set_ylabel("Density"); ax.legend(fontsize=8)
save(fig, "fig5_4_conf_by_expert", "Distribution of teacher confidence in the given v1 label, grouped by the experts' v2 decision.")

# ---------- Fig 5.5 PlantDoc P2 teacher confusion matrix ----------
P = np.load(f"{FEAT}/teacher_probs/pd_p2_role_dino+clip.npy").astype(np.float32)
te = (pdm.p2_role == "test").values; yt = pdm.class_id.values[te]; yp = P[te].argmax(1)
names = pdm.drop_duplicates("class_id").sort_values("class_id").class_name.values
cm = confusion_matrix(yt, yp, labels=range(len(names)), normalize="true")
fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=6)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=6)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.colorbar(im, fraction=.04)
save(fig, "fig5_5_pd_teacher_confusion", "Row-normalised confusion matrix of the DINOv2+CLIP teacher on the PlantDoc clean test set (P2).")

pd.DataFrame(log).to_csv(LOG, index=False)
print("\nFigures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))
print("Step 5.6 DONE.")

Relabel rate | conflict: 0.309 | non-conflict: 0.005

=== NOISE DETECTION (corrected) ===
 dataset                 comparison     n  pos  AUROC     AP   base
     pw         relabelled vs kept  6647  349 0.8329 0.2986 0.0525
     pw            removed vs kept 10236 3938 0.6542 0.5549 0.3847
     pw relabelled|removed vs kept 10585 4287 0.6687 0.5938 0.4050
     pw     label_conflict vs rest 18542 1130 0.7567 0.1954 0.0609
     pd     label_conflict vs rest  2576   94 0.7691 0.1101 0.0365
saved fig5_1_teacher_probes
saved fig5_2_leakage_inflation
saved fig5_3_noise_roc
saved fig5_4_conf_by_expert
saved fig5_5_pd_teacher_confusion

Figures: ['fig5_1_teacher_probes.png', 'fig5_2_leakage_inflation.png', 'fig5_3_noise_roc.png', 'fig5_4_conf_by_expert.png', 'fig5_5_pd_teacher_confusion.png']
Step 5.6 DONE.


In [7]:
# ===== Step 5.7: dataset-audit figures (Dataset chapter) =====
import numpy as np, pandas as pd, os, glob, re, shutil, matplotlib.pyplot as plt
from PIL import Image
FEAT, FIG = "/kaggle/working/feats", "/kaggle/working/figures"
PD_ROOT = "/kaggle/input/datasets/mashfii/plantdoc-clean-thesis"
PW_ROOT = "/kaggle/input/datasets/mashfii/plantwild-clean-thesis"
PROC = {"pd": f"{PD_ROOT}/proc", "pw": f"{PW_ROOT}/proc"}
plt.rcParams.update({"font.size": 9, "savefig.bbox": "tight"})
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG).to_dict("records") if os.path.exists(LOG) else []
def save(fig, name, cap):
    for e in ["png", "pdf"]: fig.savefig(f"{FIG}/{name}.{e}", dpi=300)
    plt.close(fig); log[:] = [r for r in log if r["file"] != name] + [dict(file=name, step="dataset", caption=cap)]
    print("saved", name)
def img_of(tag, p):
    p = str(p); i = p.find("/proc/"); return os.path.join(PROC[tag], p[i+6:]) if i >= 0 else p
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
NAME = {"pd": "PlantDoc", "pw": "PlantWild v1"}

# ---------- D1/D2 class distribution: official vs clean ----------
for tag, fs in [("pd", (7, 7)), ("pw", (7, 15))]:
    df = M[tag]; c = df.groupby("class_name").agg(
        off_train=("p1_role", lambda s: s.isin(["train", "val"]).sum()), off_test=("p1_role", lambda s: (s == "test").sum()),
        cl_train=("p2_role", lambda s: s.isin(["train", "val"]).sum()), cl_test=("p2_role", lambda s: (s == "test").sum()))
    c = c.sort_values("off_train"); y = np.arange(len(c))
    fig, ax = plt.subplots(figsize=fs)
    ax.barh(y + .2, c.off_train, .4, color="#C0C0C0", label="official train+val")
    ax.barh(y + .2, c.off_test, .4, left=c.off_train, color="#808080", label="official test")
    ax.barh(y - .2, c.cl_train, .4, color="#4C72B0", label="clean train+val")
    ax.barh(y - .2, c.cl_test, .4, left=c.cl_train, color="#DD8452", label="clean test")
    ax.set_yticks(y); ax.set_yticklabels(c.index, fontsize=6 if tag == "pw" else 7)
    ax.set_xlabel("Images"); ax.legend(fontsize=7, loc="lower right"); ax.set_title(f"{NAME[tag]}: class distribution")
    c.to_csv(f"{FIG}/table_class_counts_{tag}.csv")
    save(fig, f"figD{1 if tag=='pd' else 2}_class_dist_{tag}", f"{NAME[tag]} per-class image counts, official vs cleaned splits.")

# ---------- D3 small-image shortcut ----------
def is_healthy(n):
    n = n.lower().replace("_", " ")
    return ("healthy" in n) or bool(re.fullmatch(r"[a-z]+( [a-z]+)? leaf", n)) and not any(
        k in n for k in ["rust", "blight", "spot", "mildew", "scab", "rot", "virus", "mold", "mosaic"])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 2.6]})
for ax, tag in zip(axs, ["pd", "pw"]):
    df = M[tag]; s = df.groupby("class_name").small_flag.mean().sort_values(ascending=False)
    h = np.array([is_healthy(n) for n in s.index])
    print(f"{tag} healthy classes detected ({h.sum()}):", list(s.index[h]))
    ax.bar(range(len(s)), 100*s.values, color=np.where(h, "#55A868", "#C44E52"))
    dz, hz = 100*df.small_flag[~df.class_name.map(is_healthy)].mean(), 100*df.small_flag[df.class_name.map(is_healthy)].mean()
    ax.set_title(f"{NAME[tag]}: disease {dz:.1f}% vs healthy {hz:.1f}% small"); ax.set_xticks([])
    ax.set_xlabel("Classes (sorted)"); ax.set_ylabel("% small images")
axs[1].bar([], [], color="#C44E52", label="disease"); axs[1].bar([], [], color="#55A868", label="healthy"); axs[1].legend()
save(fig, "figD3_small_image_shortcut", "Share of small images per class; low resolution concentrates in disease classes (a resolution shortcut).")

# ---------- D4 threshold sensitivity ----------
fig, ax = plt.subplots(figsize=(5.2, 3.3)); plotted = False
for tag, pat in [("pd", "plantdoc_threshold_sensitivity.csv"), ("pw", "plantwild_v1_threshold_sensitivity.csv")]:
    f = glob.glob(f"{PW_ROOT}/**/{pat}", recursive=True)
    if not f: print("missing", pat); continue
    t = pd.read_csv(f[0]); print(f"\n{pat} columns:", list(t.columns)); print(t.head(8).to_string())
    lab = [c for c in t.columns if t[c].dtype == object]; pc = [c for c in t.columns if "leak" in c.lower() and "%" in c or "pct" in c.lower()]
    if lab and pc:
        ax.plot(t[lab[0]].astype(str), t[pc[0]], "o-", label=NAME[tag]); plotted = True
if plotted:
    ax.set_ylabel("Official test with twin in train (%)"); ax.tick_params(axis="x", rotation=35); ax.legend(); ax.grid(alpha=.3)
    save(fig, "figD4_threshold_sensitivity", "Test-to-train leak rate across duplicate-detection settings (strict hash/embedding only, plus ORB inlier thresholds).")
else: plt.close(fig); print("D4 skipped - send columns above")

# ---------- D5 protocol composition ----------
cols = [("pd", c) for c in ["p1_role", "p2_role", "p3_f0_role", "xd_pd2pw_role"]] + [("pw", c) for c in ["p1_role", "p2_role", "xd_pw2pd_role"]]
cols = [(t, c) for t, c in cols if c in M[t]]
roles, colr = ["train", "val", "test", "excluded"], ["#4C72B0", "#55A868", "#DD8452", "#C0C0C0"]
fig, ax = plt.subplots(figsize=(8, 3.4)); comp = []
for i, (t, c) in enumerate(cols):
    vc = M[t][c].fillna("excluded").value_counts(); left = 0
    for r, cc in zip(roles, colr):
        v = vc.get(r, 0); ax.barh(i, v, left=left, color=cc, label=r if i == 0 else None); left += v
    comp.append(dict(dataset=t, protocol=c, **{r: int(vc.get(r, 0)) for r in roles}))
ax.set_yticks(range(len(cols))); ax.set_yticklabels([f"{NAME[t]} · {c.replace('_role','')}" for t, c in cols])
ax.set_xlabel("Images"); ax.legend(ncol=4, fontsize=8); ax.set_xscale("log")
pd.DataFrame(comp).to_csv(f"{FIG}/table_protocols.csv", index=False)
save(fig, "figD5_protocols", "Split composition of each evaluation protocol (log scale). P3 shows fold 0; other folds are equal in size.")

# ---------- D6 expert decisions vs our conflict flag ----------
pw = M["pw"]; j = pw[pw.expert_status.isin(["kept", "removed", "relabelled"])]
grp = np.where(j.drop_reason == "label_conflict", "Flagged conflict", "Not flagged")
ct = pd.crosstab(grp, j.expert_status, normalize="index")[["kept", "removed", "relabelled"]]
fig, ax = plt.subplots(figsize=(6, 2.4)); left = np.zeros(len(ct))
for st, cc in zip(ct.columns, ["#4C72B0", "#8C8C8C", "#C44E52"]):
    ax.barh(ct.index, ct[st], left=left, color=cc, label=st)
    for k, v in enumerate(ct[st]): ax.text(left[k] + v/2, k, f"{100*v:.1f}%", ha="center", va="center", color="white", fontsize=8)
    left += ct[st].values
ax.set_xlim(0, 1); ax.set_xlabel("Share of images"); ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(.5, 1.35))
ct.to_csv(f"{FIG}/table_expert_vs_conflict.csv")
save(fig, "figD6_expert_vs_conflict", "Expert v1→v2 decisions on the 58 shared disease classes, split by our label-conflict flag.")

# ---------- D7-D9 example pair grids ----------
def grid(pairs, name, cap, title):
    if not pairs: print("no pairs for", name); return
    fig, axs = plt.subplots(2, len(pairs), figsize=(2.1*len(pairs), 4.6))
    axs = np.array(axs).reshape(2, -1)
    for k, (a, b) in enumerate(pairs):
        for r, (tag, row) in enumerate([a, b]):
            ax = axs[r, k]; ax.imshow(Image.open(img_of(tag, row.proc_path)).convert("RGB")); ax.axis("off")
            role = row.get("p1_role", "")
            ax.set_title(f"{NAME[tag].split()[0]} · {role}\n{row.class_name[:26]}", fontsize=6.5)
    fig.suptitle(title, fontsize=10); save(fig, name, cap)

rng = np.random.default_rng(42)
def pairs_from(tag, kind, n=6):
    df = M[tag][M[tag].dup_group.notna()]; out = []
    for gid, g in df.groupby("dup_group"):
        if len(g) < 2: continue
        te, tr = g[g.p1_role == "test"], g[g.p1_role.isin(["train", "val"])]
        if kind == "leak" and len(te) and len(tr) and te.class_name.iloc[0] == tr.class_name.iloc[0]:
            out.append(((tag, te.iloc[0]), (tag, tr.iloc[0])))
        elif kind == "conflict" and g.class_name.nunique() > 1:
            a = g.iloc[0]; b = g[g.class_name != a.class_name].iloc[0]; out.append(((tag, a), (tag, b)))
        elif kind == "within" and len(tr) >= 2 and tr.class_name.nunique() == 1:
            out.append(((tag, tr.iloc[0]), (tag, tr.iloc[1])))
    idx = rng.permutation(len(out))[:n]; return [out[i] for i in idx]

grid(pairs_from("pd", "leak", 3) + pairs_from("pw", "leak", 3), "figD7_leak_twins",
     "Examples of official-test images (top) with near-duplicate twins in the training split (bottom).", "Test ↔ train leak twins")
grid(pairs_from("pd", "conflict", 3) + pairs_from("pw", "conflict", 3), "figD8_label_conflicts",
     "Near-duplicate pairs carrying different class labels (label conflicts).", "Label-conflict pairs")
grid(pairs_from("pd", "within", 3) + pairs_from("pw", "within", 3), "figD9_within_train_dups",
     "Near-duplicate pairs within the training split.", "Within-train duplicates")

# ---------- D10 cross-dataset twins ----------
try:
    f = glob.glob(f"{PW_ROOT}/**/pd_pw_cross_duplicates.csv", recursive=True)[0]
    X = pd.read_csv(f); print("\npd_pw_cross_duplicates columns:", list(X.columns)); print(X.head(3).to_string())
    pdc = [c for c in X.columns if "uid" in c.lower() and "pd" in c.lower()]
    pwc = [c for c in X.columns if "uid" in c.lower() and "pw" in c.lower()]
    pdi, pwi = M["pd"].set_index("uid"), M["pw"].set_index("uid")
    X = X[X[pdc[0]].isin(pdi.index) & X[pwc[0]].isin(pwi.index)].sample(6, random_state=1)
    pairs = [(("pd", pdi.loc[a].assign(uid=a) if False else pdi.loc[a]), ("pw", pwi.loc[b])) for a, b in zip(X[pdc[0]], X[pwc[0]])]
    grid(pairs, "figD10_cross_dataset_twins", "PlantDoc images (top) with near-duplicates in PlantWild (bottom); labels may differ.",
         "Cross-dataset twins")
except Exception as e: print("D10 skipped:", repr(e), "- send the columns printed above")

# ---------- copy tables ----------
for pat in ["audit_summary_table.csv", "pd_pw_class_mapping.csv"]:
    f = glob.glob(f"{PW_ROOT}/**/{pat}", recursive=True)
    if f: shutil.copy(f[0], f"{FIG}/table_{pat}"); print("copied", pat)
pd.DataFrame(log).to_csv(LOG, index=False)
print("\nAll figures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))
print("Step 5.7 DONE.")

saved figD1_class_dist_pd
saved figD2_class_dist_pw
pd healthy classes detected (10): ['Apple leaf', 'Cherry leaf', 'Peach leaf', 'Bell_pepper leaf', 'Blueberry leaf', 'Raspberry leaf', 'Strawberry leaf', 'Soyabean leaf', 'Tomato leaf', 'grape leaf']
pw healthy classes detected (30): ['tobacco leaf', 'peach leaf', 'bean leaf', 'cucumber leaf', 'ginger leaf', 'plum leaf', 'eggplant leaf', 'maple leaf', 'grape leaf', 'celery leaf', 'cabbage leaf', 'banana leaf', 'squash leaf', 'coffee leaf', 'soybean leaf', 'cherry leaf', 'bell pepper leaf', 'apple leaf', 'broccoli leaf', 'blueberry leaf', 'corn leaf', 'potato leaf', 'rice leaf', 'raspberry leaf', 'cauliflower leaf', 'basil leaf', 'strawberry leaf', 'garlic leaf', 'lettuce leaf', 'tomato leaf']
saved figD3_small_image_shortcut

plantdoc_threshold_sensitivity.csv columns: ['setting', 'test_leak', 'test_leak_%', 'conflict_imgs', 'clean_test']
                setting  test_leak  test_leak_%  conflict_imgs  clean_test
0  strict only (no ORB)

In [8]:
# ===== Save Step 5 outputs =====
import shutil, os
shutil.make_archive("/kaggle/working/wildleaf_step5", "zip", "/kaggle/working", ".",
                    ) if False else None
os.system("cd /kaggle/working && zip -qr wildleaf_step5.zip feats figures")
print(os.path.getsize("/kaggle/working/wildleaf_step5.zip")/1e6, "MB")
print(sorted(os.listdir("/kaggle/working/figures")))

173.401613 MB
['fig5_1_teacher_probes.pdf', 'fig5_1_teacher_probes.png', 'fig5_2_leakage_inflation.pdf', 'fig5_2_leakage_inflation.png', 'fig5_3_noise_roc.pdf', 'fig5_3_noise_roc.png', 'fig5_4_conf_by_expert.pdf', 'fig5_4_conf_by_expert.png', 'fig5_5_pd_teacher_confusion.pdf', 'fig5_5_pd_teacher_confusion.png', 'figD10_cross_dataset_twins.pdf', 'figD10_cross_dataset_twins.png', 'figD1_class_dist_pd.pdf', 'figD1_class_dist_pd.png', 'figD2_class_dist_pw.pdf', 'figD2_class_dist_pw.png', 'figD3_small_image_shortcut.pdf', 'figD3_small_image_shortcut.png', 'figD4_threshold_sensitivity.pdf', 'figD4_threshold_sensitivity.png', 'figD5_protocols.pdf', 'figD5_protocols.png', 'figD6_expert_vs_conflict.pdf', 'figD6_expert_vs_conflict.png', 'figD7_leak_twins.pdf', 'figD7_leak_twins.png', 'figD8_label_conflicts.pdf', 'figD8_label_conflicts.png', 'figD9_within_train_dups.pdf', 'figD9_within_train_dups.png', 'figure_log.csv', 'table_audit_summary_table.csv', 'table_class_counts_pd.csv', 'table_class_co